### <u>1. Objectifs du notebook</u>

- **Chargement et jointure des tables** : fusion des CSV en un seul dataset
- **Cibles** : deux variables à prédire (top 10 et position d'arrivée)
- **Retrait des fuites** : éviter que le modèle triche en utilisant les données futures
- **Feature engineering** : transformer les données brutes en features exploitables
- **Contrôles et sauvegarde** : doublons, valeurs manquantes, export du dataset final

<br/>

### <u>2. Choix des données</u>
- **Tables utilisées (7)** : results (table centrale), races, status, drivers, constructors, circuits, qualifying
- **Tables écartées** : lap_times et pit_stops (pendant la course), standings (classements après la course), constructor_results, sprint_results et seasons (redondants ou trop partiels)
- **Période** : 2006 et après (à partir de cette date le focntionnement des qualifs est homogène)

<br/>

### <u> 3. Résultat attendu : f1-table.csv</u>

- **Organisation** : une ligne par pilote et par course (+2006)
- **Informations** : uniquement des infos connues avant le départ
- **Données parasites** : aucun doublon, aucune colonne post-course


In [ ]:
# ÉTAPE 1 : CHARGEMENT ET JOINTURES

# Import
import pandas as pd


# On ne garde que les colonnes qui nous intéressent (et valeur manquante = \N)
results = pd.read_csv("../data/raw-data/results.csv", na_values="\\N")
races = pd.read_csv("../data/raw-data/races.csv", na_values="\\N")[["raceId", "year", "round", "circuitId", "date"]]
status = pd.read_csv("../data/raw-data/status.csv")
drivers = pd.read_csv("../data/raw-data/drivers.csv", na_values="\\N")[["driverId", "driverRef", "dob"]]
constructors = pd.read_csv("../data/raw-data/constructors.csv")[["constructorId", "constructorRef"]]
circuits = pd.read_csv("../data/raw-data/circuits.csv")[["circuitId", "circuitRef"]]
quali = pd.read_csv("../data/raw-data/qualifying.csv", na_values="\\N")[["raceId", "driverId", "position", "q1", "q2", "q3"]]
quali = quali.rename(columns={"position": "quali_position"})


# Jointure des 7 tables en un unique dataset 
df = results.merge(races, on="raceId")
df = df.merge(status, on="statusId")
df = df.merge(drivers, on="driverId")
df = df.merge(constructors, on="constructorId")
df = df.merge(circuits, on="circuitId")
df = df.merge(quali, on=["raceId", "driverId"], how="left")   # left : pas de qualif avant 1994


# Conversion des colonnes texte en date 
df["date"] = pd.to_datetime(df["date"])
df["dob"] = pd.to_datetime(df["dob"])


# Tests (nombre et affichages)
print(len(results), len(df))   # les deux nombres doivent être égaux
df.head()

In [ ]:
# ÉTAPE 2 : CIBLES ET RETRAIT DES FUITES

# Cibles 1 : top 10 (création colonne binaire) 
df["top10"] = (df["positionOrder"] <= 10).astype(int)       


# Cible 2 : position d'arrivée (colonne positionOrder déjà présente dans les features)


# Outil pour calculer si le pilote a abandonné
df["pilot_abandon"] = (~df["status"].str.match(r"^(Finished|\+\d+ Laps?|Lapped)$")).astype(int)


# Colonnes interdites, parce qu’elles décrivent le résultat de la course
FUTUR = ["position", "positionText", "points", "laps", "time", "milliseconds", "fastestLap", "rank", "fastestLapTime", "fastestLapSpeed", "statusId", "status", "pilot_abandon"]


# Tri chronologique des résultats 
df = df.sort_values(["date", "raceId", "positionOrder"]).reset_index(drop=True)


# Vérification : il faut que par course on ait 10 pilotes dans le top 10
print(df.groupby("raceId")["top10"].sum().value_counts())


# ANALYSE
# 37 courses ont plus de 10 pilotes dans le top 10 : elles datent toutes de 1950-1960 (voitures partagées entre pilotes)
# Elles sont exclues de la table finale (2006 et après) : aucun impact

In [ ]:
# ÉTAPE 3 : CRÉATION ET CHOIX DES FEATURES

# DESCRIPTION DES FEATURES
# 1. driver_avg_position_last5         = position moyenne sur les 5 dernières courses
# 2. driver_dnf_rate_last10            = taux d'abandon sur les 10 dernières courses
# 3. driver_season_points_before_race  = points marqués dans la saison avant la course
# 4. driver_career_starts              = nombre de courses déjà disputées (expérience)
# 5. driver_avg_position_on_circuit    = position moyenne passée sur ce circuit

# 6. team_avg_position_last5         = position moyenne des 2 voitures sur les 5 dernières courses
# 7. team_dnf_rate_last10            = taux d'abandon de l'écurie sur les 10 dernières courses
# 8. team_season_points_before_race  = points de l'écurie dans la saison avant la course

# 9. grid_position_clean             = position sur la grille (0 = stands -> dernière place)
# 10. grid_gap_to_teammate           = écart de position sur la grille avec le coéquipier
# 11. quali_gap_to_best_q1_percent   = écart au meilleur temps de Q1, en %
# 12. quali_position                 = position en qualification


# 5 Features en lien avec le Pilote
pilote = df.groupby("driverId")
df["driver_avg_position_last5"] = pilote["positionOrder"].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
df["driver_dnf_rate_last10"] = pilote["pilot_abandon"].transform(lambda s: s.shift(1).rolling(10, min_periods=1).mean())
df["driver_season_points_before_race"] = df.groupby(["driverId", "year"])["points"].transform(lambda s: s.shift(1).cumsum()).fillna(0)
df["driver_career_starts"] = pilote.cumcount()
df["driver_avg_position_on_circuit"] = df.groupby(["driverId", "circuitId"])["positionOrder"].transform(lambda s: s.shift(1).expanding().mean())


# 3 Features en lien avec l'Écurie
ecurie = df.groupby(["constructorRef", "raceId", "year"], sort=False).agg(pos=("positionOrder", "mean"),dnf=("pilot_abandon", "mean"),pts=("points", "sum")).reset_index()
ecurie["team_avg_position_last5"] = ecurie.groupby("constructorRef")["pos"].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
ecurie["team_dnf_rate_last10"] = ecurie.groupby("constructorRef")["dnf"].transform(lambda s: s.shift(1).rolling(10, min_periods=1).mean())
ecurie["team_season_points_before_race"] = ecurie.groupby(["constructorRef", "year"])["pts"].transform(lambda s: s.shift(1).cumsum()).fillna(0)
df = df.merge(
    ecurie[["constructorRef", "raceId", "team_avg_position_last5", "team_dnf_rate_last10", "team_season_points_before_race"]],
    on=["constructorRef", "raceId"],
    how="left"
)


# 3 Features en lien avec la grille et les qualifications
df["grid_position_clean"] = df["grid"].replace(0, 25)  # 0 = départ des stands -> dernière place
df["grid_gap_to_teammate"] = df["grid_position_clean"] - df.groupby(["raceId", "constructorRef"])["grid_position_clean"].transform("mean")  # écart au coéquipier
q = df["q1"].str.extract(r"(\d+):(\d+\.\d+)").astype(float)  # "1:23.456" -> minutes, secondes
q1 = q[0] * 60 + q[1]  # temps Q1 en secondes
df["quali_gap_to_best_q1_percent"] = (q1 / q1.groupby(df["raceId"]).transform("min") - 1).clip(upper=0.10)  # écart au meilleur temps de Q1, en %